# 11 · SAR

Template para el modelo autoregresivo espacial implementado en `SpatialAutoregressiveModel`.

## Hipótesis del modelo

- Parte del precio se explica por dependencia espacial entre propiedades cercanas.
- El parámetro espacial `rho` debería capturar interacción local no absorbida por las covariables.
- La interpretación principal viene por coeficientes globales, `rho` y patrones de residuos.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import geopandas as gpd
import seaborn as sns


PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from ml_core.models.sarModel import SpatialAutoregressiveModel
from ml_core.evaluation.modelEvaluator import regression_metrics
from ml_core.visualization.mapper import *
from ml_core import load_model_config, save_model_config

OUTPUT_DIR = PROJECT_ROOT / "notebooks" / "output" / "11_sar"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
sns.set_theme(style="whitegrid")


## Datos y split

In [ ]:
DATA_PATH = PROJECT_ROOT / "data" / "splits"

subset_config = {
    "train_n": 2500,
    "val_n": 800,
    "test_n": 800,
    "random_state": 42,
}


def subset_split(gdf, n_rows, random_state):
    if n_rows is None or len(gdf) <= n_rows:
        return gdf.copy().reset_index(drop=True)
    return (
        gdf.sample(n=n_rows, random_state=random_state)
        .sort_index()
        .reset_index(drop=True)
    )


train_raw = pd.read_csv(DATA_PATH / "arg_venta_data_train.csv")
gdf_train_full = gpd.GeoDataFrame(
    train_raw,
    geometry=gpd.points_from_xy(
        train_raw["longitud"],
        train_raw["latitud"]
    ),
    crs="EPSG:4326"
)

test_raw = pd.read_csv(DATA_PATH / "arg_venta_data_test.csv")
gdf_test_full = gpd.GeoDataFrame(
    test_raw,
    geometry=gpd.points_from_xy(
        test_raw["longitud"],
        test_raw["latitud"]
    ),
    crs="EPSG:4326"
)

val_raw = pd.read_csv(DATA_PATH / "arg_venta_data_val.csv")
gdf_val_full = gpd.GeoDataFrame(
    val_raw,
    geometry=gpd.points_from_xy(
        val_raw["longitud"],
        val_raw["latitud"]
    ),
    crs="EPSG:4326"
)

gdf_train = subset_split(gdf_train_full, subset_config["train_n"], subset_config["random_state"])
gdf_val = subset_split(gdf_val_full, subset_config["val_n"], subset_config["random_state"] + 1)
gdf_test = subset_split(gdf_test_full, subset_config["test_n"], subset_config["random_state"] + 2)

target_col = "log_precio"
coord_cols = ["longitud", "latitud"]
feature_cols = [
    'area_m2_total',
    'area_m2_descubierta',
    'ambientes',
    'antiguedad',
    'expensas',
    'banos',
    'cocheras',
    'estado_num',
    'disposicion_Frente',
    'disposicion_Contrafrente',
    'disposicion_Lateral'
]

print(
    f"Subset SAR -> train={len(gdf_train)} / val={len(gdf_val)} / test={len(gdf_test)}"
)


In [ ]:
X_train = gdf_train[feature_cols]
y_train = gdf_train[target_col]
coords_train = gdf_train[coord_cols].to_numpy()

X_test = gdf_test[feature_cols]
y_test = gdf_test[target_col]
coords_test = gdf_test[coord_cols].to_numpy()

X_val = gdf_val[feature_cols]
y_val = gdf_val[target_col]
coords_val = gdf_val[coord_cols].to_numpy()


## Entrenamiento

In [ ]:
sar_config = {
    "k": 5,
    "radius_km": 3.0,
}

sar_config


## Sensibilidad de hiperparámetros

Acá conviene comparar distintos valores de `k` y, si querés, distintas definiciones de vecindad.

In [ ]:
config_path = PROJECT_ROOT / "notebooks" / "cache" / "sar_best_config.json"
saved_config = load_model_config(config_path)

if saved_config is None:
    tuning_model = SpatialAutoregressiveModel(sar_config=sar_config.copy())
    tuning_model.tune_hyperparameters(
        X_train,
        y_train.values.reshape(-1, 1),
        coords_train,
    )

    save_model_config(
        tuning_model,
        config_path,
        extra={
            "target": target_col,
            "features": feature_cols,
            "subset_config": subset_config,
        },
    )

    saved_config = load_model_config(config_path)

sar_config = saved_config.get("sar_params", sar_config)
selected_k = saved_config.get("selected_k", None)

best_config = {
    "sar_params": sar_config,
    "selected_k": selected_k,
}
best_config


## Evaluación global

In [ ]:
sar_config = best_config["sar_params"]
selected_k = best_config["selected_k"]

model = SpatialAutoregressiveModel(sar_config=sar_config)
if selected_k is not None:
    model.k = int(selected_k)
    model.sar_params["k"] = int(selected_k)

model.fit(X_train, y_train, coords_train)
model

y_pred_log = model.predict(
    X_val,
    coords_val
)

# revertir log
y_pred = np.exp(np.asarray(y_pred_log).reshape(-1))
y_true = np.exp(np.asarray(y_val).reshape(-1))

metrics = regression_metrics(
    y_true,
    y_pred
)

metrics


## Visualización

In [ ]:
barrios_path = PROJECT_ROOT / 'GeoData' / 'barrios.geojson'

df_grid, barrios, std = generar_grid_predicciones(
    model,
    gdf_val,
    feature_cols
)

mapa = MapaPrecio(df_grid, barrios)

mapa.plot()

#mapa.save("mapa_modelo_lgwr.png")

#mapa.save("mapa_modelo_lgwr.pdf")



## Interpretación

A diferencia de GWR, SAR tiene una interpretación más global:

- coeficientes globales
- magnitud y signo de `rho`
- sensibilidad al número de vecinos
- concentración espacial de residuos

In [ ]:
# coef_frame = pd.DataFrame({
#     "feature": ["intercept"] + feature_cols + ["rho"],
#     "coefficient": np.asarray(model.model_.betas).reshape(-1),
# })
# coef_frame

## Residuos y export

In [ ]:
# test_export = test_df[[target_col] + coord_cols].copy()
# test_export = test_export.rename(columns={target_col: "y_true"})
# test_export["y_pred"] = np.asarray(y_pred).reshape(-1)
# test_export["residual"] = test_export["y_true"] - test_export["y_pred"]
# test_export["split"] = "test"
# test_export.to_parquet(OUTPUT_DIR / "test_predictions.parquet", index=False)
# coef_frame.to_parquet(OUTPUT_DIR / "interpretability.parquet", index=False)
# (OUTPUT_DIR / "metrics.json").write_text(json.dumps(metrics, indent=2, ensure_ascii=False))
# (OUTPUT_DIR / "run_config.json").write_text(json.dumps(sar_config, indent=2, ensure_ascii=False))